In [ ]:
import numpy as np
import xgboost as xgb
import joblib
import pickle
import os
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [ ]:
def train_nids_xgboost():
    # ----------------------------------------------------
    # STEP 0: LOAD DATA (try GAN-balanced first, fall back to original)
    # ----------------------------------------------------
    target_encoder = joblib.load('attack_classes.pkl')
    
    if os.path.exists('X_train_balanced.npy') and os.path.exists('y_train_balanced.npy'):
        print("[*] Loading BALANCED Training Matrices (Real + GAN Synthetic Data)...")
        X_train = np.load('X_train_balanced.npy')
        y_train = np.load('y_train_balanced.npy')
    else:
        print("[!] GAN-balanced data not found. Falling back to ORIGINAL processed data...")
        print("[!] (Run train_gan.ipynb for class-balanced augmentation)")
        X_train = np.load('X_train_processed.npy')
        y_train = np.load('y_train.npy')
    
    print("[*] Loading Unseen, Purely Real Testing Matrices for Rigorous Validation...")
    X_test = np.load('X_test_processed.npy')
    y_test = np.load('y_test.npy')

    num_classes = len(target_encoder.classes_)
    print(f"[+] Data loaded successfully. Processing {num_classes} distinct network traffic profiles.")
    print(f"    Training Dimensions: {X_train.shape[0]} samples, {X_train.shape[1]} features")
    print(f"    Testing Dimensions:  {X_test.shape[0]} samples, {X_test.shape[1]} features")
    
    # Show class distribution
    classes, counts = np.unique(y_train, return_counts=True)
    print(f"\n    Training Class Distribution:")
    for cls, cnt in zip(classes, counts):
        name = target_encoder.inverse_transform([cls])[0]
        print(f"      {name}: {cnt} samples")

    # ----------------------------------------------------
    # STEP 1: CONFIGURE OPTIMIZED MULTI-CLASS XGBOOST
    # ----------------------------------------------------
    print("\n[*] Configuring Extreme Gradient Boosting Hyperparameters...")
    
    model = xgb.XGBClassifier(
        n_estimators=300,
        max_depth=7,
        learning_rate=0.08,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='multi:softprob',
        num_class=num_classes,
        eval_metric='mlogloss',
        random_state=42,
        tree_method='hist'
    )

    # ----------------------------------------------------
    # STEP 2: TRAIN THE ENGINE
    # ----------------------------------------------------
    print("[*] Fitting boosted tree pathways (Training engine active)...")
    print()
    
    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=25
    )
    print("\n[+] Model optimization convergence complete.")

    # ----------------------------------------------------
    # STEP 3: EVALUATION
    # ----------------------------------------------------
    print("\n[*] Evaluating engine capability on unseen real-world traffic flows...")
    y_pred = model.predict(X_test)
    
    accuracy = accuracy_score(y_test, y_pred)
    class_names = [str(cls) for cls in target_encoder.classes_]
    
    print("\n" + "="*60)
    print(f"   SYSTEM VALIDATION PERFORMANCE: OVERALL ACCURACY = {accuracy*100:.2f}%")
    print("="*60)
    print()
    print(classification_report(y_test, y_pred, target_names=class_names))
    print("="*60)
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    print()
    print("   CONFUSION MATRIX (rows=true, cols=predicted):")
    print("   " + "-"*50)
    # Header
    header = "          " + " ".join([f"{str(c):>8s}" for c in class_names])
    print(header)
    for i, row in enumerate(cm):
        row_str = " ".join([f"{val:8d}" for val in row])
        print(f"   {class_names[i]:>8s} {row_str}")
    print("   " + "-"*50)

    # ----------------------------------------------------
    # STEP 4: EXPORT MODEL
    # ----------------------------------------------------
    print()
    output_model_path = "xgb_nids_engine.json"
    print(f"[*] Serializing production deployment weight layers to '{output_model_path}'...")
    model.save_model(output_model_path)
    
    with open('model.pkl', 'wb') as f:
        pickle.dump(model, f)
    print("[+] Model exported as 'xgb_nids_engine.json' and 'model.pkl'.")
    print("[+] SUCCESS: The offline AI core training engine is fully finalized.")
    
    return model, target_encoder

if __name__ == "__main__":
    model, target_encoder = train_nids_xgboost()